In [1]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print(os.getcwd())

/Users/dikshantmahlawat/Downloads/funnel-analysis


In [2]:
import pandas as pd

df = pd.read_csv('data/raw/customer_journey.csv', parse_dates=['Timestamp'])
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum())
print(df['PageType'].value_counts())

(12719, 10)
['SessionID', 'UserID', 'Timestamp', 'PageType', 'DeviceType', 'Country', 'ReferralSource', 'TimeOnPage_seconds', 'ItemsInCart', 'Purchased']
SessionID             0
UserID                0
Timestamp             0
PageType              0
DeviceType            0
Country               0
ReferralSource        0
TimeOnPage_seconds    0
ItemsInCart           0
Purchased             0
dtype: int64
PageType
home            5000
product_page    3987
cart            1599
checkout        1123
confirmation    1010
Name: count, dtype: int64


In [3]:
FUNNEL_ORDER = ['home', 'product_page', 'cart', 'checkout', 'confirmation']

def is_prefix(pages_set):
    in_order = [p for p in FUNNEL_ORDER if p in pages_set]
    return set(FUNNEL_ORDER[:len(in_order)]) == pages_set

session_pages = df.groupby('SessionID')['PageType'].apply(set)
violations = (~session_pages.apply(is_prefix)).sum()
print("Order violations:", violations)

for col in ['DeviceType', 'Country', 'ReferralSource', 'UserID']:
    print(col, (df.groupby('SessionID')[col].nunique() > 1).sum())

Order violations: 0
DeviceType 0
Country 0
ReferralSource 0
UserID 0


In [4]:
STEP_IDX = {s: i for i, s in enumerate(FUNNEL_ORDER)}
df['StepOrder'] = df['PageType'].map(STEP_IDX)

session_df = df.sort_values('StepOrder').groupby('SessionID').agg(
    DeviceType=('DeviceType', 'first'),
    Country=('Country', 'first'),
    ReferralSource=('ReferralSource', 'first'),
    Purchased=('Purchased', 'max'),
    max_step_idx=('StepOrder', 'max'),
    Timestamp=('Timestamp', 'min'),
).reset_index()

print(session_df.shape)
print(session_df.head())

(5000, 7)
      SessionID DeviceType  Country ReferralSource  Purchased  max_step_idx  \
0     session_0    Desktop    India   Social Media          0             0   
1     session_1     Tablet  Germany          Email          0             1   
2    session_10     Tablet    India         Direct          0             1   
3   session_100    Desktop      USA          Email          0             1   
4  session_1000     Mobile   France          Email          1             4   

            Timestamp  
0 2025-01-20 22:53:34  
1 2025-02-26 12:57:10  
2 2025-05-17 22:11:37  
3 2025-04-02 11:24:36  
4 2025-08-25 09:34:15  


In [5]:
overall = pd.DataFrame({'step': FUNNEL_ORDER, 'step_order': range(5)})
overall['sessions'] = overall['step_order'].apply(lambda i: (session_df['max_step_idx'] >= i).sum())
overall['pct_of_home'] = (overall['sessions'] / overall['sessions'].iloc[0] * 100).round(2)
overall['step_over_step_pct'] = (overall['sessions'] / overall['sessions'].shift(1) * 100).round(2)
overall['drop_off_pct'] = (100 - overall['step_over_step_pct']).round(2)
print(overall.to_string(index=False))

        step  step_order  sessions  pct_of_home  step_over_step_pct  drop_off_pct
        home           0      5000       100.00                 NaN           NaN
product_page           1      3987        79.74               79.74         20.26
        cart           2      1599        31.98               40.11         59.89
    checkout           3      1123        22.46               70.23         29.77
confirmation           4      1010        20.20               89.94         10.06


In [6]:
from scipy.stats import chi2_contingency

at_risk = session_df[session_df['max_step_idx'] >= 1].copy()
at_risk['proceeded_to_cart'] = at_risk['max_step_idx'] >= 2

for seg in ['DeviceType', 'Country', 'ReferralSource']:
    ct = pd.crosstab(at_risk[seg], at_risk['proceeded_to_cart'])
    chi2, p, dof, _ = chi2_contingency(ct)
    print(f"{seg}: chi2={chi2:.3f} p={p:.4f}")

DeviceType: chi2=0.019 p=0.9905
Country: chi2=0.840 p=0.9910
ReferralSource: chi2=3.053 p=0.3835


In [7]:
from scipy.stats import ttest_ind

pp_rows = df[df['PageType'] == 'product_page'][['SessionID', 'TimeOnPage_seconds']].merge(
    session_df[['SessionID', 'max_step_idx']], on='SessionID')
pp_rows['proceeded'] = pp_rows['max_step_idx'] >= 2

summary = pp_rows.groupby('proceeded')['TimeOnPage_seconds'].agg(['count', 'mean', 'median'])
print(summary.round(1))

t, p = ttest_ind(pp_rows[pp_rows['proceeded']]['TimeOnPage_seconds'],
                  pp_rows[~pp_rows['proceeded']]['TimeOnPage_seconds'], equal_var=False)
print(f"\nt={t:.3f}, p={p:.4f}")

           count  mean  median
proceeded                     
False       2388  97.7    99.0
True        1599  96.1    96.0

t=-1.011, p=0.3119


In [8]:
dim_session = session_df.rename(columns={'max_step_idx': 'MaxStepOrder'})
dim_session['furthest_step'] = dim_session['MaxStepOrder'].map(lambda i: FUNNEL_ORDER[i])
dim_session.to_csv('data/processed/dim_session.csv', index=False)

fact_pageviews = df[['SessionID', 'PageType', 'StepOrder', 'Timestamp', 'TimeOnPage_seconds', 'ItemsInCart']]
fact_pageviews.to_csv('data/processed/fact_pageviews.csv', index=False)

overall.to_csv('data/processed/funnel_summary.csv', index=False)

rows = []
for dim in ['DeviceType', 'Country', 'ReferralSource']:
    for seg_val, g in session_df.groupby(dim):
        home_n = (g['max_step_idx'] >= 0).sum()
        for i, step in enumerate(FUNNEL_ORDER):
            n = (g['max_step_idx'] >= i).sum()
            rows.append({'dimension': dim, 'segment_value': seg_val, 'step': step,
                         'step_order': i, 'sessions': n, 'pct_of_segment_home': round(n/home_n*100, 2)})
pd.DataFrame(rows).to_csv('data/processed/segment_summary.csv', index=False)

import os
print("Exported to data/processed/:")
for f in sorted(os.listdir('data/processed')):
    print(' -', f)

Exported to data/processed/:
 - dim_session.csv
 - fact_pageviews.csv
 - funnel_summary.csv
 - segment_summary.csv


In [9]:
import sqlite3

conn = sqlite3.connect('data/processed/funnel.db')
pd.read_csv('data/raw/customer_journey.csv').to_sql('raw_events', conn, if_exists='replace', index=False)

with open('sql/funnel_analysis.sql') as f:
    sql_script = f.read()

# split into the two statements: the view (setup), then the actual query
view_sql, query_sql = sql_script.split("WITH steps(step_order, step) AS (", 1)
conn.executescript(view_sql)

result = pd.read_sql("WITH steps(step_order, step) AS (" + query_sql, conn)
print(result)

   step_order          step  sessions  pct_of_home  step_over_step_pct
0           0          home      5000       100.00                 NaN
1           1  product_page      3987        79.74               79.74
2           2          cart      1599        31.98               40.11
3           3      checkout      1123        22.46               70.23
4           4  confirmation      1010        20.20               89.94
